In [1]:
!pip install pytorchvideo torchvision av

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 57.9 MB/s eta 0:00:00
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=8c2c8d3b21a10ccc33d095f261369e876212ef4ee00d8f1f286b19548098310e
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=ca5583265cc7db4dd34a37262cba891763df80c631817bf85edd50d7127ac7a8
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel f

In [2]:
# --- 0. THE PYTORCHVIDEO BUG FIX (Monkey Patch) ---
import sys
import torchvision.transforms.functional as F_vision
sys.modules['torchvision.transforms.functional_tensor'] = F_vision

# --- 1. IMPORTS ---
import os
import shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive

from torchvision.transforms import Compose, Lambda, Resize
from torchvision.transforms._transforms_video import NormalizeVideo
from pytorchvideo.transforms import ApplyTransformToKey, UniformTemporalSubsample
from pytorchvideo.data.encoded_video import EncodedVideo

# --- 2. CONFIGURATION ---
# Connect to Google Drive
drive.mount('/content/drive', force_remount=True)

# UPDATED: We now have 3 action classes!
CLASSES = ['cow_eat', 'cow_rest', 'cow_walk_stand']
NUM_FRAMES = 16  # X3D defaults to 16 frames
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- THE MAGIC THRESHOLD ---
# Anything below 70% confidence gets sorted into Miscellaneous
THRESHOLD = 0.50

# --- THE PATHS ---
# UPDATED: Pointing to the new action model and the singular cow test folder
MODEL_WEIGHTS_PATH = "/content/drive/MyDrive/Trained_Models/cow_x3d_eat_rest_walk_TrainedModel.pth"
TEST_FOLDER_PATH = "/content/drive/MyDrive/Test/cow_singular"
RESULT_FOLDER_PATH = "/content/drive/MyDrive/Result"

# UPDATED: Mapping the new guesses to your new folders
BUCKET_NAMES = {
    'cow_eat': 'Predicted_cow_eat',
    'cow_rest': 'Predicted_cow_rest',
    'cow_walk_stand': 'Predicted_cow_walk_stand',
    'miscellaneous': 'Predicted_Miscellaneous'
}

# --- 3. BUILD THE ROBOT BODY & LOAD THE BRAIN ---
print("Building the robot (X3D-M) and loading the brain...")
model = torch.hub.load('facebookresearch/pytorchvideo', 'x3d_m', pretrained=False)

# Change the head so it knows we have 3 classes now
in_features = model.blocks[5].proj.in_features
model.blocks[5].proj = nn.Linear(in_features, len(CLASSES))

model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# --- 4. VIDEO GLASSES (TRANSFORMS) ---
video_transform = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(NUM_FRAMES),
            Lambda(lambda x: x / 255.0),
            NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225]), # X3D specifics
            Resize((256, 256)) # X3D specifics
        ])
    )
])

# --- 5. THE SORTING LOOP ---
print(f"\n--- STARTING THE SORTING MACHINE (Threshold: {THRESHOLD*100}%) ---")

# Look at every single file inside the test folder
for filename in os.listdir(TEST_FOLDER_PATH):

    # Only look at video files, ignore random hidden files
    if not filename.lower().endswith(('.mp4', '.avi', '.mov')):
        continue

    video_path = os.path.join(TEST_FOLDER_PATH, filename)
    print(f"\nWatching: {filename}...")

    try:
        # Load the 2-second clip and put the glasses on it
        video = EncodedVideo.from_path(video_path)
        video_data = video.get_clip(start_sec=0.0, end_sec=2.0)
        video_data = video_transform(video_data)
        input_tensor = video_data["video"].unsqueeze(0).to(DEVICE)

        # Make the guess!
        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, 1)

        # --- THE MISCELLANEOUS INTERCEPT ---
        confidence_score = conf.item()
        original_guess = CLASSES[pred_idx.item()]

        if confidence_score < THRESHOLD:
            winning_guess = 'miscellaneous'
            print(f"Guess: MISCELLANEOUS (Unsure about {original_guess.upper()}, only {confidence_score*100:.2f}% sure)")
        else:
            winning_guess = original_guess
            print(f"Guess: {winning_guess.upper()} ({confidence_score*100:.2f}% sure)")

        # --- THE FILE MOVING MAGIC ---
        # Look up the exact folder name from your dictionary
        target_bucket = BUCKET_NAMES[winning_guess]

        # Build the paths
        target_folder = os.path.join(RESULT_FOLDER_PATH, target_bucket)
        target_path = os.path.join(target_folder, filename)

        # ⭐️ IMPORTANT: Make sure the folder exists!
        # This will automatically generate the 4 specific folders inside "Result" if they don't exist yet
        os.makedirs(target_folder, exist_ok=True)

        # Copy the video into your folder!
        shutil.copy(video_path, target_path)
        print(f"-> Safely copied into {target_bucket}!")

    except Exception as e:
        print(f"Oh no! Something went wrong with {filename}: {e}")

print("\n--- ALL DONE! CHECK YOUR RESULT FOLDERS! ---")

/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(


Streaming output truncated to the last 5000 lines.
Guess: COW_WALK_STAND (57.61% sure)
-> Safely copied into Predicted_cow_walk_stand!

Watching: apr16_t0004_f010101.mp4...
Guess: COW_REST (55.87% sure)
-> Safely copied into Predicted_cow_rest!

Watching: apr16_t0004_f010855.mp4...
Guess: MISCELLANEOUS (Unsure about COW_WALK_STAND, only 49.09% sure)
-> Safely copied into Predicted_Miscellaneous!

Watching: apr16_t0004_f011028.mp4...
Guess: COW_WALK_STAND (57.61% sure)
-> Safely copied into Predicted_cow_walk_stand!

Watching: apr16_t0004_f010732.mp4...
Guess: COW_WALK_STAND (57.61% sure)
-> Safely copied into Predicted_cow_walk_stand!

Watching: apr16_t0004_f011066.mp4...
Guess: COW_WALK_STAND (57.61% sure)
-> Safely copied into Predicted_cow_walk_stand!

Watching: apr16_t0004_f010241.mp4...
Guess: COW_WALK_STAND (57.42% sure)
-> Safely copied into Predicted_cow_walk_stand!

Watching: apr16_t0004_f011653.mp4...
Guess: COW_WALK_STAND (57.61% sure)
-> Safely copied into Predicted_cow_wal